# Pathway Enrichment Analysis

This notebook performs pathway and GO enrichment analysis:
- GO enrichment (BP, MF, CC)
- KEGG pathway enrichment
- Gene Set Enrichment Analysis (GSEA)

**Input**: Gene list (DE genes) or ranked gene list
**Outputs**: Enrichment plots and tables

In [ ]:
# Parameters
PVALUE_CUTOFF = 0.05
TOP_TERMS = 20
ORGANISM = 'human'  # human, mouse
GENE_ID_TYPE = 'symbol'  # symbol, ensembl, entrez
FIGURE_DPI = 150

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from galaxy_notebook import get, put, list_inputs

try:
    import gseapy as gp
    GSEAPY_AVAILABLE = True
except ImportError:
    GSEAPY_AVAILABLE = False
    print("Warning: gseapy not installed, GSEA analysis unavailable")

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = FIGURE_DPI

In [ ]:
# List available inputs
print("Available inputs:")
for inp in list_inputs():
    print(f"  [{inp['index']}] {inp['name']}")

In [ ]:
# Load gene data
df = get(0)

# Detect gene column
gene_cols = ['gene', 'gene_id', 'Gene', 'symbol', 'gene_symbol', 'gene_name']
gene_col = None
for col in gene_cols:
    if col in df.columns:
        gene_col = col
        break

if gene_col is None:
    gene_col = df.columns[0]

# Check if this is a ranked list (has log2FC or similar)
rank_cols = ['log2FoldChange', 'logFC', 'log2FC', 'stat', 'score']
rank_col = None
for col in rank_cols:
    if col in df.columns:
        rank_col = col
        break

print(f"Gene column: {gene_col}")
print(f"Rank column: {rank_col}")
print(f"Total genes: {len(df)}")
df.head()

In [ ]:
# Prepare gene list
genes = df[gene_col].dropna().unique().tolist()

# If we have ranks, prepare ranked list
if rank_col:
    ranked_genes = df.set_index(gene_col)[rank_col].dropna().sort_values(ascending=False)
    print(f"Ranked gene list prepared with {len(ranked_genes)} genes")
else:
    ranked_genes = None

print(f"Gene list: {len(genes)} genes")

## 1. GO Enrichment Analysis

In [ ]:
# Run GO enrichment using Enrichr
if GSEAPY_AVAILABLE and len(genes) > 0:
    organism_map = {'human': 'Human', 'mouse': 'Mouse'}
    org = organism_map.get(ORGANISM.lower(), 'Human')
    
    # GO Biological Process
    go_bp = gp.enrichr(
        gene_list=genes[:500],  # Limit for API
        gene_sets='GO_Biological_Process_2021',
        organism=org,
        outdir=None,
        cutoff=PVALUE_CUTOFF
    )
    
    # GO Molecular Function
    go_mf = gp.enrichr(
        gene_list=genes[:500],
        gene_sets='GO_Molecular_Function_2021',
        organism=org,
        outdir=None,
        cutoff=PVALUE_CUTOFF
    )
    
    # GO Cellular Component
    go_cc = gp.enrichr(
        gene_list=genes[:500],
        gene_sets='GO_Cellular_Component_2021',
        organism=org,
        outdir=None,
        cutoff=PVALUE_CUTOFF
    )
    
    print(f"GO BP terms enriched: {len(go_bp.results[go_bp.results['Adjusted P-value'] < PVALUE_CUTOFF])}")
    print(f"GO MF terms enriched: {len(go_mf.results[go_mf.results['Adjusted P-value'] < PVALUE_CUTOFF])}")
    print(f"GO CC terms enriched: {len(go_cc.results[go_cc.results['Adjusted P-value'] < PVALUE_CUTOFF])}")
else:
    print("Skipping GO analysis - gseapy not available or no genes")

In [ ]:
# Plot GO enrichment
def plot_enrichment(results, title, ax, color='#3498db'):
    """Plot enrichment bar chart."""
    if results is None or len(results) == 0:
        ax.text(0.5, 0.5, 'No significant terms', ha='center', va='center')
        ax.set_title(title)
        return
    
    # Get top terms
    top = results.head(TOP_TERMS).copy()
    top['neg_log_p'] = -np.log10(top['Adjusted P-value'].clip(lower=1e-50))
    
    # Truncate long term names
    top['Term_short'] = top['Term'].str[:50].str.replace(r'\s*\([^)]*\)$', '', regex=True)
    
    # Plot
    y_pos = range(len(top))
    ax.barh(y_pos, top['neg_log_p'], color=color, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top['Term_short'], fontsize=8)
    ax.set_xlabel('-log10(Adjusted P-value)')
    ax.set_title(title)
    ax.invert_yaxis()

if GSEAPY_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(18, 8))
    
    plot_enrichment(go_bp.results[go_bp.results['Adjusted P-value'] < PVALUE_CUTOFF], 
                   'GO Biological Process', axes[0], '#3498db')
    plot_enrichment(go_mf.results[go_mf.results['Adjusted P-value'] < PVALUE_CUTOFF], 
                   'GO Molecular Function', axes[1], '#e74c3c')
    plot_enrichment(go_cc.results[go_cc.results['Adjusted P-value'] < PVALUE_CUTOFF], 
                   'GO Cellular Component', axes[2], '#2ecc71')
    
    plt.tight_layout()
    put(fig, 'go_enrichment.png')
    plt.show()

## 2. KEGG Pathway Enrichment

In [ ]:
# Run KEGG pathway enrichment
if GSEAPY_AVAILABLE and len(genes) > 0:
    kegg_db = 'KEGG_2021_Human' if ORGANISM.lower() == 'human' else 'KEGG_2019_Mouse'
    
    kegg = gp.enrichr(
        gene_list=genes[:500],
        gene_sets=kegg_db,
        organism=org,
        outdir=None,
        cutoff=PVALUE_CUTOFF
    )
    
    kegg_sig = kegg.results[kegg.results['Adjusted P-value'] < PVALUE_CUTOFF]
    print(f"KEGG pathways enriched: {len(kegg_sig)}")
    
    if len(kegg_sig) > 0:
        fig, ax = plt.subplots(figsize=(10, 8))
        plot_enrichment(kegg_sig, 'KEGG Pathway Enrichment', ax, '#9b59b6')
        plt.tight_layout()
        put(fig, 'kegg_enrichment.png')
        plt.show()
else:
    print("Skipping KEGG analysis")

## 3. Gene Set Enrichment Analysis (GSEA)

In [ ]:
# Run GSEA if we have ranked genes
if GSEAPY_AVAILABLE and ranked_genes is not None and len(ranked_genes) > 100:
    print("Running GSEA...")
    
    # Run prerank GSEA
    gsea_results = gp.prerank(
        rnk=ranked_genes,
        gene_sets='KEGG_2021_Human' if ORGANISM.lower() == 'human' else 'KEGG_2019_Mouse',
        outdir=None,
        min_size=15,
        max_size=500,
        permutation_num=100,
        seed=42,
        verbose=False
    )
    
    gsea_sig = gsea_results.res2d[gsea_results.res2d['FDR q-val'] < PVALUE_CUTOFF]
    print(f"Significant GSEA pathways: {len(gsea_sig)}")
    
    # Plot GSEA results
    if len(gsea_sig) > 0:
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Get top pathways by NES
        top_up = gsea_sig.nlargest(TOP_TERMS // 2, 'NES')
        top_down = gsea_sig.nsmallest(TOP_TERMS // 2, 'NES')
        top_gsea = pd.concat([top_up, top_down]).sort_values('NES')
        
        # Color by direction
        colors = ['#e74c3c' if x < 0 else '#3498db' for x in top_gsea['NES']]
        
        y_pos = range(len(top_gsea))
        ax.barh(y_pos, top_gsea['NES'], color=colors, edgecolor='white')
        ax.set_yticks(y_pos)
        ax.set_yticklabels(top_gsea['Term'].str[:40], fontsize=8)
        ax.set_xlabel('Normalized Enrichment Score (NES)')
        ax.set_title('GSEA Results')
        ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
        
        plt.tight_layout()
        put(fig, 'gsea_results.png')
        plt.show()
else:
    print("Skipping GSEA - no ranked gene list available or insufficient genes")

## 4. Combined Dot Plot

In [ ]:
# Create combined dot plot
if GSEAPY_AVAILABLE:
    # Combine results
    all_results = []
    
    for name, res in [('GO BP', go_bp), ('GO MF', go_mf), ('GO CC', go_cc), ('KEGG', kegg)]:
        if res is not None and len(res.results) > 0:
            top = res.results[res.results['Adjusted P-value'] < PVALUE_CUTOFF].head(5).copy()
            top['Category'] = name
            all_results.append(top)
    
    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        combined['neg_log_p'] = -np.log10(combined['Adjusted P-value'].clip(lower=1e-50))
        combined['Gene_count'] = combined['Overlap'].str.extract(r'(\d+)').astype(float)
        combined['Term_short'] = combined['Term'].str[:40]
        
        fig, ax = plt.subplots(figsize=(12, 10))
        
        scatter = ax.scatter(
            combined['neg_log_p'],
            range(len(combined)),
            s=combined['Gene_count'] * 10,
            c=pd.Categorical(combined['Category']).codes,
            cmap='Set2',
            alpha=0.7,
            edgecolors='white'
        )
        
        ax.set_yticks(range(len(combined)))
        ax.set_yticklabels(combined['Term_short'], fontsize=8)
        ax.set_xlabel('-log10(Adjusted P-value)')
        ax.set_title('Combined Enrichment Overview')
        
        # Add category legend
        for i, cat in enumerate(combined['Category'].unique()):
            ax.scatter([], [], c=f'C{i}', label=cat, s=100)
        ax.legend(title='Category', loc='lower right')
        
        plt.tight_layout()
        put(fig, 'enrichment_dotplot.png')
        plt.show()

## 5. Summary

In [ ]:
# Generate summary
summary = f"""
# Pathway Enrichment Analysis Summary

## Input
- Total genes analyzed: {len(genes)}
- Organism: {ORGANISM}
- P-value cutoff: {PVALUE_CUTOFF}

## GO Enrichment Results
"""

if GSEAPY_AVAILABLE:
    bp_sig = len(go_bp.results[go_bp.results['Adjusted P-value'] < PVALUE_CUTOFF])
    mf_sig = len(go_mf.results[go_mf.results['Adjusted P-value'] < PVALUE_CUTOFF])
    cc_sig = len(go_cc.results[go_cc.results['Adjusted P-value'] < PVALUE_CUTOFF])
    kegg_count = len(kegg.results[kegg.results['Adjusted P-value'] < PVALUE_CUTOFF])
    
    summary += f"""- Biological Process: {bp_sig} significant terms
- Molecular Function: {mf_sig} significant terms
- Cellular Component: {cc_sig} significant terms

## KEGG Pathway Results
- Significant pathways: {kegg_count}

### Top GO BP Terms
"""
    
    for i, row in go_bp.results.head(5).iterrows():
        summary += f"- {row['Term'][:60]}: p={row['Adjusted P-value']:.2e}\n"
    
    summary += "\n### Top KEGG Pathways\n"
    for i, row in kegg.results.head(5).iterrows():
        summary += f"- {row['Term'][:60]}: p={row['Adjusted P-value']:.2e}\n"

print(summary)
put(summary, 'enrichment_summary.md')

In [ ]:
# Save enrichment tables
if GSEAPY_AVAILABLE:
    go_bp.results.to_csv('/tmp/go_bp_results.tsv', sep='\t', index=False)
    put(go_bp.results, 'go_bp_results.tsv')
    put(kegg.results, 'kegg_results.tsv')

print("\n=== Analysis Complete ===")
print("Generated outputs:")
print("  - go_enrichment.png")
print("  - kegg_enrichment.png")
print("  - gsea_results.png (if ranked list)")
print("  - enrichment_dotplot.png")
print("  - enrichment_summary.md")
print("  - go_bp_results.tsv")
print("  - kegg_results.tsv")